# Export backends for the forecasters

In [338]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster
from skforecast.preprocessing import DateTimeFeatureTransformer
import joblib
import zipfile
import os

# Download data
# ==============================================================================
data = fetch_dataset(
    name="h2o", raw=True, kwargs_read={"names": ["y", "date"], "header": 0}, verbose=False
)
data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d')
data = data.set_index('date')
data = data.asfreq('MS')
data

,y
date,
1991-07-01,0.429795
1991-08-01,0.400906
1991-09-01,0.432159
1991-10-01,0.492543
1991-11-01,0.502369
...,...
2008-02-01,0.761822
2008-03-01,0.649435
2008-04-01,0.827887


In [339]:
# Fit export process
# ======================================================================================
calendar_transformer = DateTimeFeatureTransformer(
    features = ['month', 'day_of_month'],
    keep_original_columns = True
)

data = calendar_transformer.fit_transform(X=data)
data_train = data.iloc[:190, :]
data_test = data.iloc[190:, :]

forecaster = ForecasterRecursive(
                 estimator     = RandomForestRegressor(random_state=123),
                 lags          = 5,
                 forecaster_id = "forecaster_model",
                 transformer_y=StandardScaler()
             )

forecaster.fit(y=data_train['y'], exog=data_train.drop(columns='y'))
forecaster

=================== 
ForecasterRecursive 
=================== 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 123, 'verbose': 0,
    'warm_start': False} 
fit_kwargs: {} 
Creation date: 2026-06-09 18:10:10 
Last fit date: 2026-06-09 18:10:10 
Skforecast version: 0.23.0 
Python version: 3.13.12 
Forecaster id: forecaster_model

# Pickle

In [331]:
import pickle

# Save
with open("forecaster.pkl", "wb") as f:
    pickle.dump(forecaster, f)

# Load
with open("forecaster.pkl", "rb") as f:
    forecaster = pickle.load(f)

forecaster.predict(steps=1, exog=data_test)

2007-05-01    0.649723
Freq: MS, Name: pred, dtype: float64

# Cloudpickle

In [332]:
import cloudpickle

# Save
with open("forecaster.cloudpickle", "wb") as f:
    cloudpickle.dump(forecaster, f)

# Load
with open("forecaster.cloudpickle", "rb") as f:
    forecaster = cloudpickle.load(f)

forecaster.predict(steps=1, exog=data_test)

2007-05-01    0.649723
Freq: MS, Name: pred, dtype: float64

# Skops

In [333]:
forecaster.training_range_

DatetimeIndex(['1991-07-01', '2007-04-01'], dtype='datetime64[ns]', name='date', freq=None)

In [327]:
from skops.io import dump, load, get_untrusted_types

In [340]:
# Decompose last_window_ and training_range in values and index
last_window_dict = forecaster.last_window_.to_dict(orient="split")
last_window_dict["index"] = [str(ts) for ts in forecaster.last_window_.index]
last_window_dict['freq'] = forecaster.last_window_.index.freqstr
training_range_list = [str(ts) for ts in forecaster.training_range_]
forecaster.last_window_ = last_window_dict
forecaster.training_range_ = training_range_list

# Save object
dump(forecaster, "forecaster.skops")

# Load object
unknown_types = get_untrusted_types(file="forecaster.skops")
forecaster = load(file="forecaster.skops", trusted=unknown_types)

# Recostruct last_window_ and training_range
last_window_ = pd.DataFrame(
    data=forecaster.last_window_["data"], 
    index=pd.to_datetime(forecaster.last_window_["index"]),
    columns=forecaster.last_window_["columns"]
)
last_window_ = last_window_.asfreq(forecaster.last_window_["freq"])

training_range_ = pd.DatetimeIndex(forecaster.training_range_)
forecaster.last_window_ = last_window_
forecaster.training_range_ = training_range_

forecaster.predict(steps=1, exog=data_test)

2007-05-01    0.649723
Freq: MS, Name: pred, dtype: float64

In [ ]:
# Check what attributes can be serialized with skops
# ==============================================================================
import skops.io

# Assuming 'forecaster' is your trained skforecast object
print("Testing forecaster attributes for skops compatibility...\n")
print("-" * 50)

for attr_name, attr_value in vars(forecaster).items():
    try:
        # Attempt to serialize the individual attribute
        dumps(attr_value)
        print(f"✅ SUCCESS : '{attr_name}' (Type: {type(attr_value).__name__})")
    except Exception as e:
        # If it fails, catch the error and print it
        print(f"❌ FAILED  : '{attr_name}' (Type: {type(attr_value).__name__})")
        print(f"   -> Error: {type(e).__name__}: {e}")

print("-" * 50)
print("\nDiagnostic complete.")

Testing forecaster attributes for skops compatibility...

--------------------------------------------------
✅ SUCCESS : 'estimator' (Type: RandomForestRegressor)
✅ SUCCESS : 'transformer_y' (Type: StandardScaler)
✅ SUCCESS : 'transformer_exog' (Type: NoneType)
✅ SUCCESS : 'categorical_features' (Type: str)
✅ SUCCESS : 'weight_func' (Type: NoneType)
✅ SUCCESS : 'source_code_weight_func' (Type: NoneType)
✅ SUCCESS : 'differentiation' (Type: NoneType)
✅ SUCCESS : 'differentiation_max' (Type: NoneType)
✅ SUCCESS : 'differentiator' (Type: NoneType)
✅ SUCCESS : 'dropna_from_series' (Type: bool)
✅ SUCCESS : 'last_window_' (Type: dict)
✅ SUCCESS : 'index_type_' (Type: ABCMeta)
✅ SUCCESS : 'index_freq_' (Type: MonthBegin)
✅ SUCCESS : 'training_range_' (Type: list)
✅ SUCCESS : 'series_name_in_' (Type: str)
✅ SUCCESS : 'exog_in_' (Type: bool)
✅ SUCCESS : 'exog_names_in_' (Type: list)
✅ SUCCESS : 'exog_type_in_' (Type: type)
✅ SUCCESS : 'exog_dtypes_in_' (Type: dict)
✅ SUCCESS : 'exog_dtypes_out_